In [ ]:
import os
import shutil
import nibabel as nib
import numpy as np
import pandas as pd

# === Dossiers à analyser ===
root_paths = [
    "/home/amenacer/Stage/Data/Segmentation Emilien/CTRL/CTRL 2",
    "/home/amenacer/Stage/Data/Segmentation Emilien/CTRL/CTRL 4",
    "/home/amenacer/Stage/Data/Segmentation Emilien/HFS/HFS 4",
    "/home/amenacer/Stage/Data/Segmentation Emilien/HFS/HFS 11"
]

# === Dossier de sortie de la base
output_base = "/home/amenacer/Stage/base_de_donnees"
metadata_csv = os.path.join(output_base, "metadata.csv")

# === Catégories de fichiers
categories = ["2D", "2D+T", "3D", "3D+T"]
for cat in categories:
    os.makedirs(os.path.join(output_base, cat), exist_ok=True)

# === Initialisation
metadata = []

# === Parcours des dossiers
for root in root_paths:
    group = "CTRL" if "CTRL" in root else "HFS"
    patient = os.path.basename(root)

    for sub in os.listdir(root):
        if sub.startswith("output_"):
            sub_path = os.path.join(root, sub)

            for dirpath, _, files in os.walk(sub_path):
                for file in files:
                    if file.endswith(".nii") or file.endswith(".nii.gz"):
                        full_path = os.path.join(dirpath, file)
                        try:
                            img = nib.load(full_path)
                            data = img.get_fdata()
                            shape = data.shape

                            # === Squeeze (128, 128, 1, T) → (128, 128, T)
                            if len(shape) == 4 and shape[2] == 1:
                                data = np.squeeze(data, axis=2)
                                shape = data.shape
                                print(f"🔁 Squeezed {file} → {shape}")
                                img = nib.Nifti1Image(data, affine=img.affine)

                            # === Déterminer le type
                            if len(shape) == 2:
                                tag = "2D"
                            elif len(shape) == 3:
                                tag = "2D+T" if shape[2] < 10 else "3D"
                            elif len(shape) == 4:
                                tag = "3D+T"
                            else:
                                tag = "Inconnu"

                            # === Nom unique
                            unique_name = f"{group}_{patient}_{sub}_{file}"
                            dest_path = os.path.join(output_base, tag, unique_name)

                            # === Enregistrer le NIfTI (pas copier brut)
                            nib.save(img, dest_path)

                            # === Enregistrer les métadonnées
                            metadata.append({
                                "Fichier": unique_name,
                                "Groupe": group,
                                "Patient": patient,
                                "Dossier": sub,
                                "Type": tag,
                                "Dimensions": shape,
                                "Chemin source": full_path,
                                "Copie vers": dest_path
                            })

                            print(f"✅ {unique_name} → {tag} {shape}")

                        except Exception as e:
                            print(f"⚠️ Erreur avec {file} : {e}")

# === Enregistrement des métadonnées
df = pd.DataFrame(metadata)
df.to_csv(metadata_csv, index=False)
print(f"\n📄 Fichier metadata créé : {metadata_csv}")
